# 01 - Sequencing, alignment and callset quality control

![cohort](https://img.shields.io/badge/cohort-KZ--219-1b9e77)
![platform](https://img.shields.io/badge/platform-DNBSEQ--G400-7570b3)
![pipeline](https://img.shields.io/badge/DRAGEN-v3.10.16-d95f02)
![reference](https://img.shields.io/badge/reference-GRCh37%2Fhg19-e7298a)
![annotation](https://img.shields.io/badge/ANNOVAR-v2025Mar02-666666)
![bioproject](https://img.shields.io/badge/BioProject-PRJNA1224597-333333)

| | |
|---|---|
| **Cohort** | KZ-219, healthy Kazakh adults (145 XX, 74 XY by DRAGEN ploidy estimation) |
| **Sequencing** | DNBSEQ-G400, MGIEasy PCR-Free, 100 bp paired-end |
| **Alignment and variant calling** | Illumina DRAGEN Bio-IT Platform v3.10.16, per-sample gVCF mode |
| **Reference** | GRCh37 / hg19 (DRAGEN hash table v8) |
| **Raw data** | NCBI SRA, BioProject PRJNA1224597 |

### Manuscript items produced here

| Item | Content |
|---|---|
| **Supplementary Table 1** | sequencing, alignment, coverage and variant calling quality (former main-text Table 1 merged with the extended cohort table) |
| **Supplementary Fig. 2** | fragment length, GC content, read quality, depth |
| **Supplementary Data 1** | per-sample metrics, Het/Hom by chromosome, ROH flag, QC gate, DRAGEN agreement, depth correlations, Supplementary Table 1, source data for Supplementary Fig. 2, methods, session info |

### Running

Input is one directory per sample with the DRAGEN v3.10 metric files
(`<sample>/<sample>.mapping_metrics.csv`, `wgs_coverage_metrics`, `ploidy_estimation_metrics`,
`vc_metrics`, `fragment_length_hist`, `fastqc_metrics`, `vc_hethom_ratio_metrics`,
`roh_metrics`). Paths are read from environment variables, with defaults relative to the
repository root:

```bash
export KZ_DRAGEN_DIR=data/dragen      # one sub-directory per sample
export KZ_OUT_DIR=results/01_qc
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute --inplace 01_wgs_qc_metrics.ipynb
```

Per-sample DRAGEN metrics are deposited as Supplementary Data 1; raw reads are in NCBI SRA
under BioProject PRJNA1224597.

Figures are written as TIFF (CMYK, LZW, 1200 dpi) and PNG (RGB, 1200 dpi); all lettering is
Arial 8-10 pt, falling back to Liberation Sans where Arial is not installed.

## 1 - Setup

In [ ]:
import glob
import os
import platform
import re
import sys
from datetime import datetime

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

try:
    from scipy import stats as sstats
except ImportError:
    sstats = None

import openpyxl
print('python', sys.version.split()[0], '| pandas', pd.__version__, '| numpy', np.__version__,
      '| matplotlib', mpl.__version__, '| openpyxl', openpyxl.__version__,
      '| scipy', 'absent' if sstats is None else __import__('scipy').__version__)

In [ ]:
MM = 1 / 25.4
W1, W15, W2 = 88 * MM, 120 * MM, 180 * MM

FS, FS_PANEL = 8, 10
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': FS, 'axes.labelsize': FS, 'axes.titlesize': FS,
    'xtick.labelsize': FS, 'ytick.labelsize': FS, 'legend.fontsize': FS,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.edgecolor': '#3C3C3C', 'xtick.color': '#3C3C3C', 'ytick.color': '#3C3C3C',
    'text.color': '#1A1A1A', 'axes.labelcolor': '#1A1A1A',
    'axes.axisbelow': True,
    'grid.linewidth': 0.4, 'grid.color': '#C8C8C8',
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
})

INK, RULE, BAND = '#1A1A1A', '#3C3C3C', '#4A4A4A'

ANC_COLORS = {'East': '#2C6E9B', 'West': '#B5502F', 'Unassigned': '#A0A0A0'}
SERIES1, SERIES2 = '#3A4E60', '#9DB0BE'
FILL1 = '#DCE4EA'
FLAG = ANC_COLORS['West']
LS1, LS2, LS_REF = '-', (0, (4, 2)), (0, (3, 2))


def panel(ax, letter, dx=-0.10, dy=1.02):
    ax.text(dx, dy, letter, transform=ax.transAxes, fontsize=FS_PANEL, fontweight='bold',
            va='bottom', ha='left', color=INK)


def despine(ax, keep=('left', 'bottom')):
    for name, sp in ax.spines.items():
        sp.set_visible(name in keep)


def save_fig(fig, stem, outdir='.', dpi=1200, formats=('tiff', 'png')):
    """PNG (RGB, for screen and the Word file) is rendered once; the TIFF for the submission
    system is converted from it to CMYK (RGB is rejected there) with LZW compression and the
    dpi tag kept."""
    paths = []
    png = os.path.join(outdir, f'{stem}.png')
    fig.savefig(png, dpi=dpi)
    if 'tiff' in formats:
        tif = os.path.join(outdir, f'{stem}.tiff')
        with Image.open(png) as im:
            im.convert('RGB').convert('CMYK').save(tif, compression='tiff_lzw', dpi=(dpi, dpi))
        paths.append(tif)
    if 'png' in formats:
        paths.append(png)
    else:
        os.remove(png)
    for p in paths:
        with Image.open(p) as im:
            w, h, mode = *im.size, im.mode
        print(f'saved: {p}  {mode}, {w} x {h} px, {w / dpi * 25.4:.0f} x {h / dpi * 25.4:.0f} mm at {dpi} dpi, '
              f'{os.path.getsize(p) / 1e6:.1f} MB')
    return paths


def pearson_ci(x, y, alpha=0.05):
    # Pearson r with a Fisher z interval and a two-sided p value.
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, n = x[ok], y[ok], int(ok.sum())
    if n < 4:
        return dict(r=np.nan, lo=np.nan, hi=np.nan, p=np.nan, n=n)
    r = float(np.corrcoef(x, y)[0, 1])
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    if sstats is not None:
        crit = sstats.norm.ppf(1 - alpha / 2)
        t = r * np.sqrt((n - 2) / max(1 - r * r, 1e-300))
        p = float(2 * sstats.t.sf(abs(t), n - 2))
    else:
        crit, p = 1.959963985, np.nan
    return dict(r=r, lo=float(np.tanh(z - crit * se)), hi=float(np.tanh(z + crit * se)),
                p=p, n=n)


def fmt_p(p):
    if not np.isfinite(p):
        return 'p n/a'
    return 'p < 0.001' if p < 1e-3 else f'p = {p:.3f}'

## 2 - Configuration

In [ ]:
RUN_DIR = os.environ.get('KZ_DRAGEN_DIR', 'data/dragen')
OUT_DIR = os.environ.get('KZ_OUT_DIR', 'results/01_qc')

COHORT, N_EXPECTED = 'KZ-219', 219

FRAG_TRIM_BP = (50, 1000)
DPI = 1200
FIG_FORMATS = ('tiff', 'png')

QC_GATE = {
    'Depth (×)':                 ('>=', 20.0),
    'Breadth >=10× (%)':         ('>=', 90.0),
    'Q30 (%)':                   ('>=', 80.0),
    'Duplicate (%)':             ('<=', 10.0),
    'Uniformity (%)':            ('>=', 90.0),
    'Percent Callability':       ('>=', 95.0),
    'Ti/Tv':                     ('range', (1.95, 2.10)),
    'Het/Hom':                   ('range', (1.30, 2.10)),
}

METHODS = {
    'cohort': COHORT,
    'platform': 'DNBSEQ-G400, MGIEasy PCR-Free, 100 bp paired-end',
    'aligner and caller': 'Illumina DRAGEN Bio-IT Platform v3.10.16, germline gVCF mode',
    'reference': 'GRCh37 / hg19',
    'bioproject': 'PRJNA1224597',
    'duplicate handling': 'marked by DRAGEN, not removed',
    'fragment length window (bp)': f'{FRAG_TRIM_BP[0]}-{FRAG_TRIM_BP[1]}',
    'read-level denominator': 'total input reads',
    'base-level denominator': 'total bases',
    'mismatch and soft-clip denominator': 'mapped bases R1',
    'figure output': f'{", ".join(FIG_FORMATS)} at {DPI} dpi',
}

assert os.path.isdir(RUN_DIR), f'DRAGEN metrics directory not found: {RUN_DIR} (set KZ_DRAGEN_DIR)'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'cohort  : {COHORT}, expecting {N_EXPECTED} samples')
print(f'run dir : {RUN_DIR}')
print(f'out dir : {OUT_DIR}')

## 3 - DRAGEN metric parsing

DRAGEN metric CSVs are comma separated with the layout
`SECTION, subgroup, metric name, count, percentage`. The subgroup column carries a read group
name in `mapping_metrics` and the sample name in `vc_metrics`, so the two files need different
handling; only summary sections are used, and samples split across several read groups are not
counted twice.

| Metric | Denominator |
| --- | --- |
| mapped, unmapped, unique, properly paired, discordant, duplicate, MAPQ | total input reads |
| Q30 bases | total bases |
| mismatched and soft-clipped bases R1 | mapped bases R1 (the DRAGEN denominator) |
| breadth and uniformity | the full hg19 assembly, N gaps included, which is why breadth at 1× is near 97% rather than 100% |

In [ ]:
def sample_of(path):
    """DRAGEN writes one directory per sample; the directory name is the sample ID."""
    return os.path.basename(os.path.dirname(path))


def find(pattern):
    return sorted(glob.glob(os.path.join(RUN_DIR, '*', pattern)))


def parse_dragen_csv(path, keys=None, sections=('SUMMARY',), skip_read_group=True, column=3):
    """Return {metric: value} for the requested sections; column 3 is the count, 4 the percentage."""
    rec = {}
    with open(path) as fh:
        for line in fh:
            parts = [x.strip() for x in line.rstrip('\n').split(',')]
            if len(parts) < 4:
                continue
            section, subgroup, key = parts[0], parts[1], parts[2]
            if skip_read_group and subgroup:
                continue
            if sections and not any(s in section for s in sections):
                continue
            if keys is not None and key not in keys:
                continue
            value = parts[column] if len(parts) > column else ''
            rec[key] = pd.to_numeric(value, errors='coerce')
    return rec


def dump_keys(path, limit=None):
    """List every section and metric name in a DRAGEN metrics file."""
    seen = []
    with open(path) as fh:
        for line in fh:
            parts = [x.strip() for x in line.rstrip('\n').split(',')]
            if len(parts) >= 4:
                seen.append((parts[0], parts[1], parts[2]))
    out = pd.DataFrame(seen, columns=['Section', 'Subgroup', 'Metric'])
    return out.head(limit) if limit else out


def mean_sd(s, digits=2, thousands=False):
    fmt = f',.{digits}f' if thousands else f'.{digits}f'
    return f'{s.mean():{fmt}} ± {s.std():{fmt}}'


def summarise(frame, spec):
    """Build a publication table from (label, column, digits, thousands) tuples."""
    rows = []
    for label, col, digits, thou in spec:
        s = frame[col].dropna()
        fmt = f',.{digits}f' if thou else f'.{digits}f'
        rows.append({'Metric': label, 'n': int(s.shape[0]),
                     'Mean ± SD': f'{s.mean():{fmt}} ± {s.std():{fmt}}',
                     'Min': f'{s.min():{fmt}}', 'Max': f'{s.max():{fmt}}'})
    return pd.DataFrame(rows).set_index('Metric')


PATTERNS = ['mapping_metrics', 'wgs_coverage_metrics', 'ploidy_estimation_metrics',
            'vc_metrics', 'fragment_length_hist', 'fastqc_metrics',
            'vc_hethom_ratio_metrics', 'roh_metrics']
inventory = pd.DataFrame({p: pd.Series({sample_of(f): 1 for f in find(f'*.{p}.csv')})
                          for p in PATTERNS}).fillna(0).astype(int)
inventory.index.name = 'Sample'
print(f'samples with at least one metric file: {len(inventory)}')
print(inventory.sum().to_string())
incomplete = inventory[inventory.sum(axis=1) < len(PATTERNS)]
if len(incomplete):
    print(f'\nWARNING: {len(incomplete)} sample(s) missing at least one metric file')
    display(incomplete)

## 4 - Sequencing, alignment and coverage

In [ ]:
MAP_KEYS = [
    'Total input reads', 'Total bases', 'Mapped reads', 'Unmapped reads',
    'Number of unique & mapped reads (excl. duplicate marked reads)',
    'Number of duplicate marked reads', 'Properly paired reads',
    'Not properly paired reads (discordant)',
    'Q30 bases', 'Soft-clipped bases R1', 'Soft-clipped bases R2',
    'Mismatched bases R1 (excl. indels)', 'Mismatched bases R2 (excl. indels)',
    'Total bases R1', 'Total bases R2', 'Mapped bases R1', 'Mapped bases R2',
    'Insert length: mean', 'Insert length: median', 'Insert length: standard deviation',
    'Estimated read length',
    'Reads with MAPQ [40:inf)', 'Reads with MAPQ [30:40)', 'Reads with MAPQ [20:30)',
    'Reads with MAPQ [10:20)', 'Reads with MAPQ [ 0:10)',
]

rows = []
for path in find('*.mapping_metrics.csv'):
    rec = parse_dragen_csv(path, MAP_KEYS)
    rec['Sample'] = sample_of(path)
    rows.append(rec)
mapping = pd.DataFrame(rows).set_index('Sample')

rows = []
for path in find('*.wgs_coverage_metrics.csv'):
    rec = parse_dragen_csv(path, None)
    rec['Sample'] = sample_of(path)
    rows.append(rec)
coverage = pd.DataFrame(rows).set_index('Sample')

rows = []
for path in find('*.ploidy_estimation_metrics.csv'):
    rec = {'Sample': sample_of(path)}
    with open(path) as fh:
        for line in fh:
            p = [x.strip() for x in line.split(',')]
            if len(p) < 4:
                continue
            if p[2] == 'Ploidy estimation':
                rec['Sex'] = p[3]
            elif p[2] == 'Autosomal median coverage':
                rec['Autosomal median cov'] = pd.to_numeric(p[3], errors='coerce')
            elif p[2] == 'X median coverage':
                rec['X median cov'] = pd.to_numeric(p[3], errors='coerce')
            elif p[2] == 'Y median coverage':
                rec['Y median cov'] = pd.to_numeric(p[3], errors='coerce')
    rows.append(rec)
ploidy = pd.DataFrame(rows).set_index('Sample')
ploidy['X/A ratio'] = ploidy['X median cov'] / ploidy['Autosomal median cov']
ploidy['Y/A ratio'] = ploidy['Y median cov'] / ploidy['Autosomal median cov']

aln = mapping.join(coverage, how='left', rsuffix='_cov').join(ploidy, how='left')


def breadth_col(frame, threshold):
    pat = re.compile(r'coverage \[\s*' + re.escape(threshold) + r'x\s*:\s*inf\)', re.I)
    hits = [c for c in frame.columns if pat.search(c)]
    return hits[0] if hits else None


reads, bases = aln['Total input reads'], aln['Total bases']
mapped_r1 = aln['Mapped bases R1'] if 'Mapped bases R1' in aln else aln['Total bases R1']

aln['Total reads (M)'] = reads / 1e6
aln['Mapped (%)'] = aln['Mapped reads'] / reads * 100
aln['Unmapped (%)'] = aln['Unmapped reads'] / reads * 100
aln['Unique mapped (%)'] = aln['Number of unique & mapped reads (excl. duplicate marked reads)'] / reads * 100
aln['Properly paired (%)'] = aln['Properly paired reads'] / reads * 100
aln['Discordant (%)'] = aln['Not properly paired reads (discordant)'] / reads * 100
aln['Duplicate (%)'] = aln['Number of duplicate marked reads'] / reads * 100
aln['MAPQ>=40 (%)'] = aln['Reads with MAPQ [40:inf)'] / reads * 100
aln['Q30 (%)'] = aln['Q30 bases'] / bases * 100
aln['Mismatch R1 (%)'] = aln['Mismatched bases R1 (excl. indels)'] / mapped_r1 * 100
aln['Soft-clipped R1 (%)'] = aln['Soft-clipped bases R1'] / mapped_r1 * 100
aln['Insert mean (bp)'] = aln['Insert length: mean'].replace(0, np.nan)
aln['Insert median (bp)'] = aln['Insert length: median'].replace(0, np.nan)
aln['Insert SD (bp)'] = aln['Insert length: standard deviation'].replace(0, np.nan)
aln['Read length (bp)'] = aln['Estimated read length']

aln['Depth (×)'] = aln['Average alignment coverage over genome']
aln['Autosomal depth (×)'] = aln['Average autosomal coverage over genome']
aln['Median autosomal cov (×)'] = aln['Median autosomal coverage over genome']
aln['Uniformity (%)'] = aln['Uniformity of coverage (PCT > 0.2*mean) over genome']
aln['chrX depth (×)'] = aln['Average chr X coverage over genome']
aln['chrY depth (×)'] = aln['Average chr Y coverage over genome']
aln['chrM depth (×)'] = aln['Average mitochondrial coverage over genome']
for thr, label in [('1', 'Breadth >=1× (%)'), ('10', 'Breadth >=10× (%)'), ('20', 'Breadth >=20× (%)')]:
    col = breadth_col(aln, thr)
    aln[label] = aln[col] if col else np.nan

assert len(aln) == N_EXPECTED, f'parsed {len(aln)} samples, expected {N_EXPECTED}'

ALN_SPEC = [
    ('Total reads (×10⁶)', 'Total reads (M)', 1, True),
    ('Mapped reads (%)', 'Mapped (%)', 2, False),
    ('Unmapped reads (%)', 'Unmapped (%)', 2, False),
    ('Uniquely mapped reads (%)', 'Unique mapped (%)', 2, False),
    ('Properly paired reads (%)', 'Properly paired (%)', 2, False),
    ('Discordant reads (%)', 'Discordant (%)', 2, False),
    ('Duplicate marked reads (%)', 'Duplicate (%)', 2, False),
    ('Reads with MAPQ ≥ 40 (%)', 'MAPQ>=40 (%)', 2, False),
    ('Mismatched bases, R1 excl. indels (%)', 'Mismatch R1 (%)', 2, False),
    ('Soft-clipped bases, R1 (%)', 'Soft-clipped R1 (%)', 2, False),
    ('Q30 bases (%)', 'Q30 (%)', 2, False),
    ('Estimated read length (bp)', 'Read length (bp)', 0, False),
    ('Insert size, median (bp)', 'Insert median (bp)', 1, False),
    ('Insert size, mean (bp)', 'Insert mean (bp)', 1, False),
    ('Insert size, SD (bp)', 'Insert SD (bp)', 1, False),
    ('Mean depth of coverage (×)', 'Depth (×)', 2, False),
    ('Mean autosomal coverage (×)', 'Autosomal depth (×)', 2, False),
    ('Median autosomal coverage (×)', 'Median autosomal cov (×)', 2, False),
    ('Mean chrM coverage (×)', 'chrM depth (×)', 0, True),
    ('Breadth ≥ 1× (%)', 'Breadth >=1× (%)', 2, False),
    ('Breadth ≥ 10× (%)', 'Breadth >=10× (%)', 2, False),
    ('Breadth ≥ 20× (%)', 'Breadth >=20× (%)', 2, False),
    ('Uniformity, > 0.2 × mean (%)', 'Uniformity (%)', 2, False),
]

table_aln = summarise(aln, ALN_SPEC)
print(f'Sequencing, alignment and coverage metrics (n = {len(aln)})')
display(table_aln)

print('\ngenetic sex from ploidy estimation')
for sx in ['XX', 'XY']:
    m = aln['Sex'] == sx
    print(f'  {sx}: n = {int(m.sum()):3d}   '
          f'X/A median = {aln.loc[m, "X/A ratio"].median():.3f} '
          f'[{aln.loc[m, "X/A ratio"].min():.3f}, {aln.loc[m, "X/A ratio"].max():.3f}]   '
          f'Y/A median = {aln.loc[m, "Y/A ratio"].median():.3f} '
          f'[{aln.loc[m, "Y/A ratio"].min():.3f}, {aln.loc[m, "Y/A ratio"].max():.3f}]')
other = aln.loc[~aln['Sex'].isin(['XX', 'XY'])]
print(f'  other ploidy calls: {len(other)}')
assert len(other) == 0, 'a ploidy call other than XX or XY is present, resolve before submission'

d = aln['Depth (×)']
print(f'\ndepth spread: {d.min():.1f}-{d.max():.1f}×, {d.max() / d.min():.1f}-fold, '
      f'CV {d.std() / d.mean() * 100:.1f}%')

## 5 - Agreement with the DRAGEN-reported percentages

In [ ]:
PCT_KEYS = ['Mapped reads', 'Unmapped reads', 'Properly paired reads',
            'Not properly paired reads (discordant)', 'Number of duplicate marked reads',
            'Number of unique & mapped reads (excl. duplicate marked reads)',
            'Reads with MAPQ [40:inf)', 'Q30 bases', 'Soft-clipped bases R1',
            'Mismatched bases R1 (excl. indels)']

rows = []
for path in find('*.mapping_metrics.csv'):
    rec = parse_dragen_csv(path, PCT_KEYS, column=4)
    rec['Sample'] = sample_of(path)
    rows.append(rec)
reported = pd.DataFrame(rows).set_index('Sample')

CHECKS = [
    ('Mapped reads (%)', 'Mapped (%)', 'Mapped reads', 'total input reads'),
    ('Unmapped reads (%)', 'Unmapped (%)', 'Unmapped reads', 'total input reads'),
    ('Uniquely mapped reads (%)', 'Unique mapped (%)',
     'Number of unique & mapped reads (excl. duplicate marked reads)', 'total input reads'),
    ('Properly paired reads (%)', 'Properly paired (%)', 'Properly paired reads', 'total input reads'),
    ('Discordant reads (%)', 'Discordant (%)', 'Not properly paired reads (discordant)', 'total input reads'),
    ('Duplicate marked reads (%)', 'Duplicate (%)', 'Number of duplicate marked reads', 'total input reads'),
    ('Reads with MAPQ ≥ 40 (%)', 'MAPQ>=40 (%)', 'Reads with MAPQ [40:inf)', 'total input reads'),
    ('Q30 bases (%)', 'Q30 (%)', 'Q30 bases', 'total bases'),
    ('Soft-clipped bases, R1 (%)', 'Soft-clipped R1 (%)', 'Soft-clipped bases R1', 'mapped bases R1'),
    ('Mismatched bases, R1 excl. indels (%)', 'Mismatch R1 (%)',
     'Mismatched bases R1 (excl. indels)', 'mapped bases R1'),
]

rows = []
for label, own, key, denom in CHECKS:
    if key not in reported.columns:
        rows.append({'Metric': label, 'Denominator': denom, 'n': 0,
                     'Max |delta| (pp)': np.nan, 'Note': 'not reported by DRAGEN'})
        continue
    delta = (aln[own] - reported[key]).dropna()
    rows.append({'Metric': label, 'Denominator': denom, 'n': len(delta),
                 'Max |delta| (pp)': float(delta.abs().max()) if len(delta) else np.nan,
                 'Note': ''})
recheck = pd.DataFrame(rows).set_index('Metric')
display(recheck.round(4))

worst = recheck['Max |delta| (pp)'].max()
print(f'largest disagreement with the DRAGEN-reported percentage: {worst:.4f} pp')
assert not (worst > 1.0), 'a recomputed percentage uses a different denominator than DRAGEN'

## 6 - Per-genome variant calling and Supplementary Table 1

In [ ]:
VC_KEYS = [
    'Total', 'Biallelic', 'Multiallelic', 'SNPs', 'Insertions (Hom)', 'Insertions (Het)',
    'Deletions (Hom)', 'Deletions (Het)', 'Indels (Hom)', 'Indels (Het)',
    'SNP Transitions', 'SNP Transversions', 'Ti/Tv ratio',
    'Heterozygous', 'Homozygous', 'Het/Hom ratio',
    'Percent Callability', 'Percent Autosome Callability',
    'Chr X number of SNPs over genome', 'Chr Y number of SNPs over genome',
]

rows = []
for path in find('*.vc_metrics.csv'):
    rec = parse_dragen_csv(path, VC_KEYS, sections=('POSTFILTER',), skip_read_group=False)
    rec['Sample'] = sample_of(path)
    rows.append(rec)
vc = pd.DataFrame(rows).set_index('Sample')

REQUIRED = ['Total', 'SNPs', 'SNP Transitions', 'SNP Transversions', 'Heterozygous', 'Homozygous']
missing = [k for k in REQUIRED if k not in vc.columns]
if missing:
    example = find('*.vc_metrics.csv')[0]
    print(f'missing keys: {missing}')
    display(dump_keys(example))
    raise KeyError(f'{missing} not found, compare against the listing above')


def get(frame, *names):
    for n in names:
        if n in frame.columns:
            return frame[n]
    return pd.Series(np.nan, index=frame.index)


vc['Ti/Tv'] = vc['SNP Transitions'] / vc['SNP Transversions']
vc['Het/Hom'] = vc['Heterozygous'] / vc['Homozygous']
vc['Insertions'] = get(vc, 'Insertions (Hom)').fillna(0) + get(vc, 'Insertions (Het)').fillna(0)
vc['Deletions'] = get(vc, 'Deletions (Hom)').fillna(0) + get(vc, 'Deletions (Het)').fillna(0)
vc['Indels'] = vc['Insertions'] + vc['Deletions']
vc['Other (MNV/complex)'] = vc['Total'] - vc['SNPs'] - vc['Indels']
vc['Biallelic (%)'] = vc['Biallelic'] / vc['Total'] * 100
vc['Multiallelic (%)'] = vc['Multiallelic'] / vc['Total'] * 100
vc['SNPs (%)'] = vc['SNPs'] / vc['Total'] * 100
vc['Indels (%)'] = vc['Indels'] / vc['Total'] * 100

vc = vc.join(aln[['Sex', 'Depth (×)']], how='left')
assert len(vc) == N_EXPECTED, f'parsed {len(vc)} samples, expected {N_EXPECTED}'

for col in ['Percent Callability', 'Percent Autosome Callability']:
    if col not in vc.columns:
        print(f'note: {col} absent, the row will be blank')
        vc[col] = np.nan

VC_SPEC = [
    ('Total variants per genome', 'Total', 0, True),
    ('SNVs per genome', 'SNPs', 0, True),
    ('Indels per genome', 'Indels', 0, True),
    ('Insertions per genome', 'Insertions', 0, True),
    ('Deletions per genome', 'Deletions', 0, True),
    ('MNV and complex records per genome', 'Other (MNV/complex)', 0, True),
    ('SNVs (%)', 'SNPs (%)', 2, False),
    ('Indels (%)', 'Indels (%)', 2, False),
    ('Biallelic variants (%)', 'Biallelic (%)', 2, False),
    ('Multiallelic variants (%)', 'Multiallelic (%)', 2, False),
    ('Heterozygous genotypes', 'Heterozygous', 0, True),
    ('Homozygous alternate genotypes', 'Homozygous', 0, True),
    ('Het/Hom ratio', 'Het/Hom', 3, False),
    ('Ti/Tv ratio', 'Ti/Tv', 3, False),
    ('Callability, genome-wide (%)', 'Percent Callability', 2, False),
    ('Callability, autosomal (%)', 'Percent Autosome Callability', 2, False),
]

table_vc = summarise(vc, VC_SPEC)
print(f'Per-genome variant calling metrics (n = {len(vc)})')
display(table_vc)

print('\nTi/Tv reported by DRAGEN vs recomputed from transitions and transversions')
if 'Ti/Tv ratio' in vc:
    print(f'  reported  : {mean_sd(vc["Ti/Tv ratio"], 3)}  '
          f'[{vc["Ti/Tv ratio"].min():.3f}, {vc["Ti/Tv ratio"].max():.3f}]')
print(f'  recomputed: {mean_sd(vc["Ti/Tv"], 3)}  '
      f'[{vc["Ti/Tv"].min():.3f}, {vc["Ti/Tv"].max():.3f}]')

if 'Chr Y number of SNPs over genome' in vc:
    print('\nchrY SNV counts by ploidy call, an independent check on the sex assignment')
    for sx in ['XX', 'XY']:
        y = vc.loc[vc['Sex'] == sx, 'Chr Y number of SNPs over genome'].dropna()
        if len(y):
            print(f'  {sx}: n = {len(y)}, range [{y.min():,.0f}, {y.max():,.0f}]')

rng_tot = vc['Total'].max() - vc['Total'].min()
print(f'\nper-genome variant count range: {rng_tot:,.0f} ({rng_tot / vc["Total"].mean() * 100:.1f}% of the mean), '
      f'CV {vc["Total"].std() / vc["Total"].mean() * 100:.2f}%')

# Depth correlations: reported in the text, no figure; the numbers go to Supplementary Data 1.
rows = []
for col, name in [('Ti/Tv', 'Ti/Tv'), ('Het/Hom', 'Het/Hom'), ('Total', 'Total variants'),
                  ('Multiallelic (%)', 'Multiallelic (%)')]:
    st = pearson_ci(vc['Depth (×)'], vc[col])
    rows.append({'Metric against depth': name, 'n': st['n'], 'Pearson r': round(st['r'], 3),
                 '95% CI low': round(st['lo'], 3), '95% CI high': round(st['hi'], 3),
                 'P (two-sided)': st['p']})
    print(f'  depth vs {name:16s}: r = {st["r"]:+.2f} [{st["lo"]:+.2f}, {st["hi"]:+.2f}], '
          f'{fmt_p(st["p"])}, n = {st["n"]}')
depth_corr = pd.DataFrame(rows).set_index('Metric against depth')
spread = vc['Ti/Tv'].max() - vc['Ti/Tv'].min()
print(f'Ti/Tv spread across the cohort: {spread:.4f}, {spread / vc["Ti/Tv"].mean() * 100:.2f}% of the mean')

### Supplementary Table 1

The former main-text Table 1 and the extended cohort table are merged into one Supplementary
Table 1, grouped into four blocks. The key rows of the old Table 1 come first in each block.
Rows that are arithmetic complements of another row (unmapped = 100 - mapped, SNV % and indel %
of the total) or near-duplicates (mean insert size next to the median, mean autosomal next to
mean genome depth, breadth at 1×, read length) stay in the per-sample sheets only.

In [ ]:
sex_rows = pd.concat([
    summarise(aln[aln['Sex'] == sx], [(f'{label}, {sx} genomes (×)', col, 2, False)])
    for label, col in [('Mean chrX coverage', 'chrX depth (×)'),
                       ('Mean chrY coverage', 'chrY depth (×)')]
    for sx in ['XX', 'XY']])

both = pd.concat([table_aln, table_vc, sex_rows])
assert not both.index.duplicated().any()

ST1_LAYOUT = [
    ('Reads and alignment', [
        'Total reads (×10⁶)', 'Mapped reads (%)', 'Duplicate marked reads (%)',
        'Reads with MAPQ ≥ 40 (%)', 'Q30 bases (%)', 'Insert size, median (bp)',
        'Uniquely mapped reads (%)', 'Properly paired reads (%)', 'Discordant reads (%)',
        'Mismatched bases, R1 excl. indels (%)', 'Soft-clipped bases, R1 (%)',
        'Insert size, SD (bp)']),
    ('Coverage', [
        'Mean depth of coverage (×)', 'Breadth ≥ 10× (%)', 'Breadth ≥ 20× (%)',
        'Uniformity, > 0.2 × mean (%)', 'Median autosomal coverage (×)',
        'Mean chrM coverage (×)']),
    ('Variant calling', [
        'Total variants per genome', 'SNVs per genome', 'Indels per genome',
        'Ti/Tv ratio', 'Het/Hom ratio', 'Callability, genome-wide (%)',
        'Insertions per genome', 'Deletions per genome', 'MNV and complex records per genome',
        'Biallelic variants (%)', 'Multiallelic variants (%)', 'Heterozygous genotypes',
        'Homozygous alternate genotypes', 'Callability, autosomal (%)']),
    ('Sex chromosome coverage', list(sex_rows.index)),
]

parts = []
for block, labels in ST1_LAYOUT:
    missing = [l for l in labels if l not in both.index]
    assert not missing, f'{block}: {missing}'
    sub = both.loc[labels].reset_index()
    sub.insert(0, 'Block', [block] + [''] * (len(sub) - 1))
    parts.append(sub)
supp_table1 = pd.concat(parts, ignore_index=True)

used = {l for _, ls in ST1_LAYOUT for l in ls}
print('rows kept only in the per-sample sheets:', sorted(set(both.index) - used))
print(f'\nSupplementary Table 1 ({len(supp_table1)} rows)')
display(supp_table1)

ST1_CAPTION = (
    'Sequencing, alignment, coverage and variant calling quality across 219 Kazakh genomes. '
    'Values are means ± standard deviations with the observed range. All percentages were '
    'recomputed for each sample from the raw DRAGEN counts: total input reads are the '
    'denominator for read-level metrics, total bases for base-level metrics and mapped read 1 '
    'bases for mismatch and soft-clip rates. Breadth and uniformity are computed over the full '
    'GRCh37 assembly. Per-sample values for every metric are provided in Supplementary Data 1. '
    'MAPQ, mapping quality; SNV, single-nucleotide variant; MNV, multinucleotide variant; '
    'Ti/Tv, transition-to-transversion ratio; Het/Hom, heterozygous-to-homozygous ratio; '
    'XX and XY, genetic sex inferred from chromosomal ploidy.')

## 7 - Fragment length, base quality and GC content

Fragment length statistics are computed from the weighted histogram. Mean, median, mode and SD
use the trimmed window set in `FRAG_TRIM_BP`; the untrimmed SD is kept in the per-sample table so
that the effect of the chimeric tail stays visible.

Quality and GC distributions are converted to a percentage of reads within each sample before
averaging across the cohort. Summing raw counts instead would weight the cohort distribution
towards the most deeply sequenced genomes.

Two different Q30 quantities appear in this notebook and they are not interchangeable:
Supplementary Table 1 reports the percentage of *bases* with quality at or above 30, while this
section reports the percentage of *reads* whose mean quality is at or above 30.

In [ ]:
frag_rows, frag_freq = [], {}
lo, hi = FRAG_TRIM_BP

for path in find('*.fragment_length_hist.csv'):
    sample = sample_of(path)
    fdf = pd.read_csv(path, comment='#')
    if fdf.empty:
        continue
    lengths = fdf['FragmentLength'].to_numpy(dtype=float)
    counts = fdf['Count'].to_numpy(dtype=float)
    order = np.argsort(lengths)
    lengths, counts = lengths[order], counts[order]

    keep = (lengths >= lo) & (lengths <= hi)
    tl, tc = lengths[keep], counts[keep]
    total_all, total_trim = counts.sum(), tc.sum()
    if total_trim <= 0:
        continue

    mean = np.average(tl, weights=tc)
    sd = np.sqrt(np.average((tl - mean) ** 2, weights=tc))
    median = tl[np.searchsorted(np.cumsum(tc), total_trim / 2)]
    mode = tl[np.argmax(tc)]
    mean_all = np.average(lengths, weights=counts)
    sd_all = np.sqrt(np.average((lengths - mean_all) ** 2, weights=counts))

    frag_rows.append({'Sample': sample, 'Mean': mean, 'Median': median, 'Mode': mode,
                      'SD': sd, 'SD untrimmed': sd_all,
                      'Pct in window': total_trim / total_all * 100})
    frag_freq[sample] = pd.Series(tc / total_trim * 100, index=tl)

frag = pd.DataFrame(frag_rows).set_index('Sample').sort_index()
assert len(frag) == N_EXPECTED, f'{len(frag)} fragment-length histograms, expected {N_EXPECTED}'

grid = np.arange(lo, hi + 1, dtype=float)
frag_matrix = pd.DataFrame(0.0, index=grid, columns=list(frag_freq))
for sample, s in frag_freq.items():
    s = s.groupby(level=0).sum()
    frag_matrix.loc[s.index.intersection(grid), sample] = s.loc[s.index.intersection(grid)].to_numpy()
frag_mean, frag_sd = frag_matrix.mean(axis=1), frag_matrix.std(axis=1)

print(f'fragment length histograms: {len(frag)}')
print(f'  mean   {mean_sd(frag["Mean"], 1)} bp')
print(f'  median {mean_sd(frag["Median"], 1)} bp')
print(f'  SD     {mean_sd(frag["SD"], 1)} bp trimmed to {lo}-{hi} bp, '
      f'{mean_sd(frag["SD untrimmed"], 1)} bp untrimmed')
print(f'  read pairs inside the window: {mean_sd(frag["Pct in window"], 2)} %')
print(f'  histogram mean vs DRAGEN insert length mean: '
      f'{frag["Mean"].mean():.1f} vs {aln["Insert mean (bp)"].mean():.1f} bp, '
      f'difference {frag["Mean"].mean() - aln["Insert mean (bp)"].mean():+.1f} bp')

fq = []
for path in find('*.fastqc_metrics.csv'):
    d = pd.read_csv(path, header=None, names=['Category', 'Read', 'Metric', 'Value'],
                    on_bad_lines='skip')
    d['Sample'] = sample_of(path)
    fq.append(d)
fq = pd.concat(fq, ignore_index=True)
fq['Value'] = pd.to_numeric(fq['Value'], errors='coerce')
assert fq['Sample'].nunique() == N_EXPECTED

q = fq[fq['Category'] == 'READ MEAN QUALITY'].copy()
q = q[q['Metric'].str.contains(r'Q\d+ Reads', na=False)]
q['Q'] = q['Metric'].str.extract(r'Q(\d+)', expand=False).astype(int)
q = q.groupby(['Sample', 'Read', 'Q'], as_index=False)['Value'].sum()
q['Frac'] = q['Value'] / q.groupby(['Sample', 'Read'])['Value'].transform('sum') * 100
q_stats = q.groupby(['Read', 'Q'])['Frac'].agg(['mean', 'std']).reset_index()

q30 = q[q['Q'] >= 30].groupby(['Sample', 'Read'])['Frac'].sum().unstack()
print(f'\nfastqc files: {fq["Sample"].nunique()}')
for read in q30.columns:
    print(f'  reads with mean quality >= Q30, {read}: {mean_sd(q30[read], 2)} %')

gc = fq[fq['Category'] == 'READ GC CONTENT'].copy()
gc['GC'] = gc['Metric'].str.extract(r'(\d+)% GC Reads', expand=False).astype(float)
gc = gc.dropna(subset=['GC'])
gc = gc.groupby(['Sample', 'GC'], as_index=False)['Value'].sum()
gc['Frac'] = gc['Value'] / gc.groupby('Sample')['Value'].transform('sum') * 100
gc_stats = gc.groupby('GC')['Frac'].agg(['mean', 'std']).reset_index()
gc_mode = gc_stats.loc[gc_stats['mean'].idxmax(), 'GC']
gc_sample_mean = gc.assign(w=gc['GC'] * gc['Frac']).groupby('Sample')['w'].sum() / 100
print(f'  GC content: mode {gc_mode:.0f}%, per-sample mean {mean_sd(gc_sample_mean, 2)} %')

## 8 - Het/Hom per chromosome and long runs of homozygosity

Chromosome X is summarised separately for XX and XY genomes. In an XY genome the
non-pseudoautosomal region is haploid, so a Het/Hom ratio taken over the whole chromosome says
nothing about heterozygosity and only reflects the size of the pseudoautosomal regions.
Chromosome Y is excluded for the same reason. The table goes to Supplementary Data 1, sheet
`02_HetHom_by_chrom` (the sheet name cited in the manuscript), not to the SI tables.

The ROH metric here is the DRAGEN large-ROH summary and is used only as a per-genome QC flag for
cryptic consanguinity. The population ROH analysis in the main text is computed separately from
the joint callset and the two sets of numbers are not comparable.

In [ ]:
rows = []
for path in find('*.vc_hethom_ratio_metrics.csv'):
    sample = sample_of(path)
    with open(path) as fh:
        for line in fh:
            if 'POSTFILTER' not in line:
                continue
            p = [x.strip() for x in line.split(',')]
            if len(p) < 4:
                continue
            field = p[2].split(maxsplit=1)
            if len(field) != 2 or field[1] != 'Het/Hom ratio':
                continue
            value = pd.to_numeric(p[3], errors='coerce')
            if np.isfinite(value):
                rows.append({'Sample': sample, 'Chromosome': field[0].replace('chr', ''),
                             'Value': value})
hethom = pd.DataFrame(rows).merge(aln[['Sex']], left_on='Sample', right_index=True, how='left')

AUTOSOMES = [str(i) for i in range(1, 23)]
auto = hethom[hethom['Chromosome'].isin(AUTOSOMES)].copy()
auto['Group'] = auto['Chromosome']
xchr = hethom[hethom['Chromosome'] == 'X'].copy()
xchr['Group'] = 'X (' + xchr['Sex'] + ')'
hh = pd.concat([auto, xchr], ignore_index=True)

ORDER = AUTOSOMES + ['X (XX)', 'X (XY)']
hethom_summary = (hh.groupby('Group')['Value']
                  .agg(Mean='mean', SD='std', Median='median',
                       IQR=lambda x: x.quantile(0.75) - x.quantile(0.25),
                       Min='min', Max='max', n='size')
                  .reindex(ORDER).dropna(how='all').reset_index()
                  .rename(columns={'Group': 'Chromosome'}))

print(f'samples with Het/Hom data: {hethom["Sample"].nunique()}')
display(hethom_summary.round(3))

ROH_LABELS = {}
rows = []
for path in find('*.roh_metrics.csv'):
    rec = {'Sample': sample_of(path)}
    with open(path) as fh:
        for line in fh:
            p = [x.strip() for x in line.split(',')]
            if len(p) < 4:
                continue
            value = pd.to_numeric(p[3], errors='coerce')
            if 'Percent SNVs in large ROH' in p[2]:
                rec['Percent SNVs in ROH'] = value
                ROH_LABELS['pct'] = p[2]
            elif 'Number of large ROH' in p[2]:
                rec['Number of ROH'] = value
                ROH_LABELS['n'] = p[2]
            elif 'Largest ROH' in p[2]:
                rec['Largest ROH (bp)'] = value
    rows.append(rec)
roh = pd.DataFrame(rows).set_index('Sample')
assert len(roh) == N_EXPECTED

# Read the size threshold out of the DRAGEN metric name instead of assuming 3 Mb.
m = re.search(r'(\d[\d ]*)', ROH_LABELS.get('n', '').split('>')[-1])
ROH_MIN_BP = int(m.group(1).replace(' ', '')) if m else 3_000_000
ROH_MIN_MB = ROH_MIN_BP / 1e6
print(f'\nDRAGEN large-ROH definition taken from the metric name: > {ROH_MIN_MB:.0f} Mb')

roh_cols = [c for c in ['Percent SNVs in ROH', 'Number of ROH', 'Largest ROH (bp)'] if c in roh]
roh_stats = roh[roh_cols].agg(['mean', 'std', 'min', 'max']).T
roh_stats.columns = ['Mean', 'SD', 'Min', 'Max']
display(roh_stats.round(3))
ROH_FLAG = 5
n_high = int((roh['Number of ROH'] >= ROH_FLAG).sum())
print(f'genomes with {ROH_FLAG} or more ROH > {ROH_MIN_MB:.0f} Mb: {n_high} / {len(roh)} '
      f'({n_high / len(roh) * 100:.1f}%)')
print(f'genomes with none: {int((roh["Number of ROH"] == 0).sum())} / {len(roh)}')

## 9 - Cohort QC gate

The acceptance criteria are declared in `QC_GATE` in the configuration cell and applied here to
every genome, so the manuscript can state the thresholds and the number of genomes that met them.

In [ ]:
gate_src = aln.join(vc[['Percent Callability', 'Ti/Tv', 'Het/Hom']], how='left')
flags = pd.DataFrame(index=gate_src.index)
for col, (op, bound) in QC_GATE.items():
    s = gate_src[col]
    if op == '>=':
        flags[f'{col} >= {bound}'] = s >= bound
    elif op == '<=':
        flags[f'{col} <= {bound}'] = s <= bound
    else:
        flags[f'{col} in {bound[0]}-{bound[1]}'] = s.between(*bound)

qc_gate = flags.assign(**{'Passed all': flags.all(axis=1)})
summary = pd.DataFrame({'Genomes failing': (~flags).sum()})
display(summary)
n_pass = int(qc_gate['Passed all'].sum())
print(f'genomes meeting every criterion: {n_pass} / {len(qc_gate)}')
if n_pass < len(qc_gate):
    display(gate_src.loc[~qc_gate['Passed all'],
                         ['Depth (×)', 'Q30 (%)', 'Duplicate (%)', 'Percent Callability',
                          'Ti/Tv', 'Het/Hom']].round(3))

## 10 - Supplementary Fig. 2

Four panels: fragment length, GC content, read mean quality, per-genome depth. Drawn at 180 mm
(double column) with Arial 8 pt lettering and 10 pt panel letters, no title inside the figure.
Read 1 and read 2 differ by line style as well as colour. Written as TIFF (CMYK, LZW) for upload and PNG (RGB)
for the Word file, both at 1200 dpi. The legend is printed below for the SI file; it describes what is drawn, not the
results, and points to the source data.

In [ ]:
fig = plt.figure(figsize=(W2, 120 * MM), layout='constrained')
gs = fig.add_gridspec(2, 2)
ax_a, ax_b = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
ax_c, ax_d = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
# a
ax = ax_a
ax.fill_between(frag_mean.index, np.maximum(frag_mean - frag_sd, 0), frag_mean + frag_sd,
                color=FILL1, lw=0, zorder=1)
ax.plot(frag_mean.index, frag_mean.values, color=SERIES1, lw=1.0, zorder=2)
ax.set_xlim(lo, 600)
ax.set_ylim(bottom=0)
ax.set_xlabel('Fragment length (bp)')
ax.set_ylabel('Read pairs (%)')
ax.grid(True, axis='y')
despine(ax)
panel(ax, 'a')
ax.text(0.03, 0.95, f'mean {frag["Mean"].mean():.0f} ± {frag["Mean"].std():.0f} bp\n'
                    f'mode {frag["Mode"].median():.0f} bp',
        transform=ax.transAxes, ha='left', va='top', color=BAND)
# b
ax = ax_b
ax.fill_between(gc_stats['GC'], np.maximum(gc_stats['mean'] - gc_stats['std'], 0),
                gc_stats['mean'] + gc_stats['std'], color=FILL1, lw=0, zorder=1)
ax.plot(gc_stats['GC'], gc_stats['mean'], color=SERIES1, lw=1.0, zorder=2)
ax.axvline(gc_sample_mean.mean(), color=FLAG, lw=0.8, ls=LS_REF, zorder=3)
ax.set_xlim(0, 100)
ax.set_ylim(bottom=0)
ax.set_xlabel('GC content (%)')
ax.set_ylabel('Reads (%)')
ax.grid(True, axis='y')
despine(ax)
panel(ax, 'b')
ax.text(0.97, 0.95, f'mean {gc_sample_mean.mean():.1f}%\nmode {gc_mode:.0f}%',
        transform=ax.transAxes, ha='right', va='top', color=BAND)
# c
ax = ax_c
for read, colour, ls in [('Read1', SERIES1, LS1), ('Read2', SERIES2, LS2)]:
    sub = q_stats[q_stats['Read'] == read].sort_values('Q')
    ax.fill_between(sub['Q'], np.maximum(sub['mean'] - sub['std'], 0), sub['mean'] + sub['std'],
                    color=colour, alpha=0.22, lw=0, zorder=1)
    ax.plot(sub['Q'], sub['mean'], color=colour, lw=1.1, ls=ls, zorder=2,
            label=f'Read {read[-1]}, Q30+ {q30[read].mean():.1f}%')
ax.set_xlim(20, 41)
ax.set_ylim(bottom=0)
ax.set_xlabel('Mean read quality (Phred)')
ax.set_ylabel('Reads (%)')
ax.grid(True, axis='y')
despine(ax)
ax.legend(frameon=False, loc='upper left', handlelength=2.2, borderpad=0.1,
          handletextpad=0.5, labelspacing=0.3)
panel(ax, 'c')
# d
ax = ax_d
depth = aln['Depth (×)'].dropna()
bins = np.arange(np.floor(depth.min()), np.ceil(depth.max()) + 1, 1)
ax.hist(depth, bins=bins, color=SERIES1, edgecolor='white', lw=0.3, zorder=2)
ax.axvline(depth.median(), color=FLAG, lw=0.9, ls=LS_REF, zorder=4)
ax.set_xlabel('Mean depth of coverage (×)')
ax.set_ylabel('Genomes')
ax.grid(True, axis='y')
despine(ax)
panel(ax, 'd')
ax.text(0.97, 0.95, f'median {depth.median():.1f}×\n'
                    f'{depth.min():.0f}-{depth.max():.0f}×, {depth.max() / depth.min():.1f}-fold',
        transform=ax.transAxes, ha='right', va='top', color=BAND)

hi_roh = roh['Number of ROH'] >= ROH_FLAG
print(f'long ROH: {int(hi_roh.sum())} of {len(roh)} genomes with >= {ROH_FLAG} runs '
      f'> {ROH_MIN_MB:.0f} Mb (max {int(roh["Number of ROH"].max())}, '
      f'max {roh["Percent SNVs in ROH"].max():.1f}% SNVs)')
save_fig(fig, 'Supplementary_Fig_2_sequencing_qc', OUT_DIR, dpi=DPI, formats=FIG_FORMATS)
plt.show()

SOURCE_SHEET = '09_SourceData_SFig2'
SFIG2_LEGEND = (
    f'Supplementary Fig. 2. Sequencing quality across the {len(aln)} Kazakh genomes. '
    f'a, Fragment length distribution: cohort mean of the per-sample distributions (line) with a '
    f'band of ±1 s.d. across samples, computed over a {lo}-{hi} bp window and shown to 600 bp. '
    f'b, Read GC content, normalized within each sample before averaging: cohort mean with a '
    f'±1 s.d. band; the dashed line marks the mean of the per-sample mean GC content. '
    f'c, Distribution of per-read mean base quality for read 1 (solid line) and read 2 (dashed '
    f'line): cohort mean with a ±1 s.d. band. Q30+ in the key is the percentage of reads with a '
    f'mean quality of at least 30. d, Per-genome mean depth of coverage in 1× bins; the dashed '
    f'line marks the cohort median. Source data are provided in Supplementary Data 1, sheet '
    f'{SOURCE_SHEET}.')
print('\n' + SFIG2_LEGEND)
print(f'\nlegend words: {len(SFIG2_LEGEND.split())} (limit 350)')

## 11 - Export

**Supplementary Data 1** workbook. Sheet names are stable, because the manuscript
cites `02_HetHom_by_chrom` by name. The workbook also carries Supplementary Table 1, the depth
correlations quoted in the Results, and the source data behind every panel of Supplementary Fig. 2.
Header row formatted, body centred.

**Supplementary Table 1** goes to a .docx fragment (Arial, editable, no shading) for assembly
into the single SI PDF.

In [ ]:
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

HEAD_FILL = PatternFill(fill_type='solid', start_color='FFDDEBF7', end_color='FFDDEBF7')
HEAD_FONT = Font(name='Arial', size=10, bold=True, color='FF1F3864')
BODY_FONT = Font(name='Arial', size=10, color='FF000000')
THIN = Side(style='thin', color='FFBFBFBF')


def style_sheet(ws, freeze='A2', center_from=1):
    """Header band, frozen panes, centred body, autosized columns."""
    ws.freeze_panes = freeze
    for cell in ws[1]:
        cell.fill = HEAD_FILL
        cell.font = HEAD_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = Border(bottom=THIN)
    ws.row_dimensions[1].height = 36
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = BODY_FONT
            cell.alignment = Alignment(
                horizontal='center' if cell.column >= center_from else 'left',
                vertical='center')
    for col in ws.iter_cols(max_col=ws.max_column):
        letter = get_column_letter(col[0].column)
        width = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[letter].width = min(max(width + 3, 12), 44)

src = []
src.append(pd.DataFrame({'Panel': 'a', 'Series': 'fragment length, % of read pairs',
                         'x': frag_mean.index, 'Mean': frag_mean.values, 'SD': frag_sd.values}))
src.append(pd.DataFrame({'Panel': 'b', 'Series': 'GC content, % of reads',
                         'x': gc_stats['GC'], 'Mean': gc_stats['mean'], 'SD': gc_stats['std']}))
for read in ['Read1', 'Read2']:
    sub = q_stats[q_stats['Read'] == read].sort_values('Q')
    src.append(pd.DataFrame({'Panel': 'c', 'Series': f'{read}, % of reads by mean quality',
                             'x': sub['Q'], 'Mean': sub['mean'], 'SD': sub['std']}))
src.append(pd.DataFrame({'Panel': 'd', 'Series': 'mean depth of coverage (×), per genome',
                         'x': depth.index, 'Mean': depth.values, 'SD': np.nan}))
source_sfig2 = pd.concat(src, ignore_index=True)
source_sfig2 = source_sfig2.rename(columns={'x': 'x (bp, % GC, Phred or sample)'})
for c in ['Mean', 'SD']:
    source_sfig2[c] = pd.to_numeric(source_sfig2[c]).round(5)

XLSX = os.path.join(OUT_DIR, 'Supplementary_Data_1_sequencing_quality.xlsx')

per_sample_aln = aln[[c for _, c, _, _ in ALN_SPEC] + ['Sex', 'X/A ratio', 'Y/A ratio',
                                                      'chrX depth (×)', 'chrY depth (×)']].copy()
per_sample_vc = vc[[c for _, c, _, _ in VC_SPEC] + ['Sex', 'Depth (×)']].copy()

methods = pd.DataFrame(list(METHODS.items()), columns=['Key', 'Value'])
session = pd.DataFrame([
    ('generated', datetime.now().strftime('%Y-%m-%d %H:%M:%S')),
    ('samples parsed', str(len(aln))),
    ('python', sys.version.split()[0]),
    ('os', platform.system()),
    ('numpy', np.__version__),
    ('pandas', pd.__version__),
    ('matplotlib', mpl.__version__),
    ('openpyxl', openpyxl.__version__),
    ('scipy', 'absent' if sstats is None else __import__('scipy').__version__),
    ('pillow', Image.__version__ if hasattr(Image, '__version__') else __import__('PIL').__version__),
], columns=['Key', 'Value'])

SHEETS = [
    ('00_PerSample_alignment', per_sample_aln, True),
    ('01_PerSample_variants', per_sample_vc, True),
    ('02_HetHom_by_chrom', hethom_summary, False),
    ('03_FragmentLength', frag, True),
    ('04_ROH_large_runs', roh, True),
    ('05_QC_gate', qc_gate, True),
    ('06_DRAGEN_agreement', recheck, True),
    ('07_SupplTable1', supp_table1, False),
    ('08_Depth_correlations', depth_corr, True),
    (SOURCE_SHEET, source_sfig2, False),
    ('10_Methods', methods, False),
    ('11_SessionInfo', session, False),
]
with pd.ExcelWriter(XLSX, engine='openpyxl') as xl:
    for name, obj, idx in SHEETS:
        obj.to_excel(xl, sheet_name=name, index=idx)
    for ws in xl.book.worksheets:
        style_sheet(ws, center_from=2)

print(f'saved {XLSX}')
for name, obj, _ in SHEETS:
    print(f'  {name}: {len(obj)} rows')

In [ ]:
from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.shared import Inches, Pt

FONT = 'Arial'

def _run(par, text, size, bold=False):
    r = par.add_run(text)
    r.bold = bold
    r.font.size = size
    r.font.name = FONT
    r._element.rPr.rFonts.set(qn('w:eastAsia'), FONT)
    return r


def add_table(doc, df, caption, tag, size=Pt(9)):
    p = doc.add_paragraph()
    _run(p, f'{tag} ', Pt(10), bold=True)
    _run(p, caption, Pt(10))
    p.paragraph_format.space_after = Pt(6)
    t = doc.add_table(rows=1, cols=len(df.columns))
    t.style = 'Table Grid'          # borders only, no shading
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for j, col in enumerate(df.columns):
        c = t.rows[0].cells[j]
        c.text = ''
        _run(c.paragraphs[0], str(col), size, bold=True)
        c.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
    for _, row in df.iterrows():
        cells = t.add_row().cells
        for j, v in enumerate(row):
            cells[j].text = ''
            _run(cells[j].paragraphs[0], '' if pd.isna(v) else str(v), size,
                 bold=(j == 0 and str(v) != ''))
            cells[j].paragraphs[0].alignment = (
                WD_ALIGN_PARAGRAPH.LEFT if j <= 1 else WD_ALIGN_PARAGRAPH.CENTER)
    doc.add_paragraph().paragraph_format.space_after = Pt(10)


doc = Document()
st = doc.styles['Normal']
st.font.name, st.font.size = FONT, Pt(10)
st.element.rPr.rFonts.set(qn('w:eastAsia'), FONT)
for s in doc.sections:
    s.left_margin = s.right_margin = Inches(0.8)

add_table(doc, supp_table1, ST1_CAPTION, 'Supplementary Table 1.')
DOCX = os.path.join(OUT_DIR, 'Supplementary_Table_1.docx')
doc.save(DOCX)
print(f'saved {DOCX}: {len(supp_table1)} rows x {supp_table1.shape[1]} cols')